# Closing Python Generators
## Advanced Tutorial Problems with Complete Solutions

This notebook develops a rigorous understanding of generator shutdown through guided, multi-step problems.

Each problem follows a tutorial pattern:

1. define the engineering situation,
2. establish the mental model,
3. divide the solution into logical steps,
4. implement incrementally,
5. test multiple exit paths,
6. explain the result,
7. extract a reusable design rule.

The focus is not merely on syntax. The focus is on **resource ownership**, **deterministic cleanup**, and **correct shutdown protocols**.

## What this notebook covers

You will work with:

- generator lifecycle states,
- `close()` and `GeneratorExit`,
- legal and illegal shutdown behavior,
- file-owning generators,
- early termination,
- `yield from`,
- close-propagating pipelines,
- context-managed iterator wrappers,
- generator coroutines,
- explicit transaction protocols,
- generator-based context managers,
- async generators and `aclose()`,
- validated multi-file CSV streams,
- systematic cleanup tests.

## Central design principle

> A generator that acquires a resource should release it in `finally`.  
> A consumer that stops before exhaustion should close the generator deterministically.

Implementing only one side of this principle is not enough.

## Notebook conventions

- Examples use only the Python standard library.
- Cleanup behavior is recorded in audit lists so it can be asserted.
- Expected errors are caught, keeping the notebook runnable from top to bottom.
- `eval()` is avoided.
- Garbage-collection timing is never used as a correctness guarantee.

In [1]:
from __future__ import annotations

import asyncio
import csv
import inspect
import itertools
import tempfile

from collections.abc import AsyncIterator, Callable, Generator, Iterable, Iterator
from contextlib import aclosing, closing, contextmanager
from dataclasses import dataclass, field
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any, TypeVar

T = TypeVar("T")
U = TypeVar("U")

print("Environment ready.")

Environment ready.


# Problem 1 — Build a generator lifecycle laboratory

## Situation

A generator can be created, started, suspended, resumed, exhausted, closed, or terminated by an exception.

Before writing resource-owning generators, you need a precise model of those transitions.

## Goal

Create one observable generator and use it to answer four questions:

1. Does calling the generator function execute its body?
2. What happens after the first `next()`?
3. Does normal exhaustion run cleanup?
4. Does closing a suspended generator run cleanup?

## Step 1 — Define an observable generator

The audit list makes internal events testable.

Notice that the `finally` block surrounds both suspension points.

In [2]:
def lifecycle_stream(audit: list[str]) -> Iterator[int]:
    audit.append("entered")
    try:
        audit.append("before-10")
        yield 10

        audit.append("after-10")
        audit.append("before-20")
        yield 20

        audit.append("after-20")
    finally:
        audit.append("cleanup")

## Step 2 — Create without advancing

Calling a generator function creates a generator object but does not enter the function body.

In [3]:
created_audit: list[str] = []
created = lifecycle_stream(created_audit)

assert inspect.getgeneratorstate(created) == "GEN_CREATED"
assert created_audit == []

print(inspect.getgeneratorstate(created), created_audit)

GEN_CREATED []


## Step 3 — Close before the body starts

Closing a never-started generator changes its state to closed, but Python does not enter the body merely to run `finally`.

That is safe because no resource inside the body was acquired.

In [4]:
created.close()

assert inspect.getgeneratorstate(created) == "GEN_CLOSED"
assert created_audit == []

print(inspect.getgeneratorstate(created), created_audit)

GEN_CLOSED []


## Step 4 — Advance once

The first `next()` enters the body and runs until the first `yield`.

In [5]:
suspended_audit: list[str] = []
suspended = lifecycle_stream(suspended_audit)

assert next(suspended) == 10
assert inspect.getgeneratorstate(suspended) == "GEN_SUSPENDED"
assert suspended_audit == ["entered", "before-10"]

print(inspect.getgeneratorstate(suspended), suspended_audit)

GEN_SUSPENDED ['entered', 'before-10']


## Step 5 — Close while suspended

Now the protected region has been entered, so closing must unwind the frame and run cleanup.

In [6]:
suspended.close()

assert inspect.getgeneratorstate(suspended) == "GEN_CLOSED"
assert suspended_audit == ["entered", "before-10", "cleanup"]

print(inspect.getgeneratorstate(suspended), suspended_audit)

GEN_CLOSED ['entered', 'before-10', 'cleanup']


## Step 6 — Exhaust normally

Natural exhaustion also runs `finally`, but all code between the suspension points executes first.

In [7]:
exhausted_audit: list[str] = []
exhausted = lifecycle_stream(exhausted_audit)

values = list(exhausted)

assert values == [10, 20]
assert exhausted_audit == [
    "entered",
    "before-10",
    "after-10",
    "before-20",
    "after-20",
    "cleanup",
]

print(values)
print(exhausted_audit)

[10, 20]
['entered', 'before-10', 'after-10', 'before-20', 'after-20', 'cleanup']


## Problem 1 takeaway

A useful lifecycle model is:

- `GEN_CREATED`: body has not started,
- `GEN_RUNNING`: body is executing,
- `GEN_SUSPENDED`: paused at `yield`,
- `GEN_CLOSED`: terminated.

For cleanup, the crucial question is whether the generator entered the region that owns the resource.

# Problem 2 — Observe the shutdown signal

## Situation

Calling `close()` does not simply set a Boolean flag.

Python raises a special exception at the point where the generator is suspended.

## Goal

Prove that:

- the signal is `GeneratorExit`,
- it inherits from `BaseException`,
- it is not caught by `except Exception`,
- `finally` still runs.

## Step 1 — Inspect the hierarchy

In [8]:
assert issubclass(GeneratorExit, BaseException)
assert not issubclass(GeneratorExit, Exception)

print(GeneratorExit.__mro__)

(<class 'GeneratorExit'>, <class 'BaseException'>, <class 'object'>)


## Step 2 — Build separate handlers

The generator records ordinary exceptions and generator shutdown separately.

In [9]:
def shutdown_probe(audit: list[str]) -> Iterator[str]:
    try:
        yield "ready"
    except Exception as exc:
        audit.append(f"ordinary:{type(exc).__name__}")
        raise
    except GeneratorExit:
        audit.append("generator-exit")
        raise
    finally:
        audit.append("finally")

## Step 3 — Close the suspended generator

Re-raising `GeneratorExit` is cooperative. Python recognizes it as the expected response to `close()`.

In [10]:
close_probe_audit: list[str] = []
close_probe = shutdown_probe(close_probe_audit)

assert next(close_probe) == "ready"
close_probe.close()

assert close_probe_audit == ["generator-exit", "finally"]
assert inspect.getgeneratorstate(close_probe) == "GEN_CLOSED"

print(close_probe_audit)

['generator-exit', 'finally']


## Step 4 — Compare with `throw()`

`throw()` injects an application exception at the suspension point.

In [11]:
throw_probe_audit: list[str] = []
throw_probe = shutdown_probe(throw_probe_audit)

assert next(throw_probe) == "ready"

try:
    throw_probe.throw(ValueError("bad input"))
except ValueError as exc:
    assert str(exc) == "bad input"

assert throw_probe_audit == ["ordinary:ValueError", "finally"]

print(throw_probe_audit)

['ordinary:ValueError', 'finally']


## Problem 2 takeaway

A generator may cooperate with close by:

- allowing `GeneratorExit` to propagate,
- catching and re-raising it,
- catching it and returning immediately.

It must not continue producing values.

# Problem 3 — Diagnose “generator ignored GeneratorExit”

## Situation

A developer attempts to yield one final status value after receiving a close request.

This violates the generator shutdown protocol.

## Goal

Create the broken design, observe the exact failure, inspect the remaining state, and replace the final yield with a safe reporting mechanism.

## Step 1 — Write the intentionally broken generator

In [12]:
def illegal_final_yield(audit: list[str]) -> Iterator[int]:
    try:
        yield 1
    except GeneratorExit:
        audit.append("caught-close")
        yield 999
    finally:
        audit.append("cleanup")

## Step 2 — Trigger the error safely

Python raises `RuntimeError` because the generator produced another value after being told to terminate.

In [13]:
illegal_audit: list[str] = []
illegal = illegal_final_yield(illegal_audit)

assert next(illegal) == 1

try:
    illegal.close()
except RuntimeError as exc:
    illegal_message = str(exc)
else:
    raise AssertionError("RuntimeError was expected")

assert illegal_message == "generator ignored GeneratorExit"
assert inspect.getgeneratorstate(illegal) == "GEN_SUSPENDED"

print(illegal_message)
print(inspect.getgeneratorstate(illegal))
print(illegal_audit)

generator ignored GeneratorExit
GEN_SUSPENDED
['caught-close']


## Step 3 — Finish unwinding

The failed close left the generator suspended at the illegal yield. A second close injects a fresh `GeneratorExit` at that new point.

In [14]:
illegal.close()

assert inspect.getgeneratorstate(illegal) == "GEN_CLOSED"
assert illegal_audit == ["caught-close", "cleanup"]

print(illegal_audit)

['caught-close', 'cleanup']


## Step 4 — Replace the final yield with shared state

Shutdown metadata belongs outside the iteration channel.

In [15]:
@dataclass
class StreamReport:
    produced: int = 0
    closed_early: bool = False


def reportable_stream(report: StreamReport) -> Iterator[int]:
    try:
        for value in range(5):
            report.produced += 1
            yield value
    except GeneratorExit:
        report.closed_early = True
        raise


report = StreamReport()
reporting = reportable_stream(report)

assert next(reporting) == 0
assert next(reporting) == 1

reporting.close()

assert report == StreamReport(produced=2, closed_early=True)
print(report)

StreamReport(produced=2, closed_early=True)


## Problem 3 takeaway

A close request ends iteration. It is not an opportunity to produce a final item.

Use an audit object, callback, shared report, or context manager for shutdown information.

# Problem 4 — Build a file-owning generator

## Situation

A file parser yields records lazily. A consumer may stop after finding one match.

The file must close after:

- normal exhaustion,
- explicit close,
- a generator-side exception,
- a consumer-side exception protected by an ownership boundary.

## Step 1 — Implement lazy acquisition and `finally` cleanup

In [16]:
def nonempty_lines(path: Path, audit: list[str]) -> Iterator[str]:
    audit.append("open")
    file = path.open("r", encoding="utf-8")

    try:
        for raw_line in file:
            line = raw_line.strip()
            if line:
                yield line
    finally:
        file.close()
        audit.append("close")

## Step 2 — Confirm lazy opening

Creating the generator should not open the file.

In [17]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "lines.txt"
    path.write_text("alpha\n\nbeta\ngamma\n", encoding="utf-8")

    lazy_file_audit: list[str] = []
    lazy_lines = nonempty_lines(path, lazy_file_audit)

    assert lazy_file_audit == []

    lazy_lines.close()
    assert lazy_file_audit == []

print("Lazy acquisition verified.")

Lazy acquisition verified.


## Step 3 — Test normal exhaustion

In [18]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "lines.txt"
    path.write_text("alpha\n\nbeta\ngamma\n", encoding="utf-8")

    full_file_audit: list[str] = []
    lines = list(nonempty_lines(path, full_file_audit))

    assert lines == ["alpha", "beta", "gamma"]
    assert full_file_audit == ["open", "close"]

print(lines, full_file_audit)

['alpha', 'beta', 'gamma'] ['open', 'close']


## Step 4 — Test explicit early close

In [19]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "lines.txt"
    path.write_text("alpha\nbeta\ngamma\n", encoding="utf-8")

    early_file_audit: list[str] = []
    early_lines = nonempty_lines(path, early_file_audit)

    assert next(early_lines) == "alpha"
    assert early_file_audit == ["open"]

    early_lines.close()

    assert early_file_audit == ["open", "close"]

print(early_file_audit)

['open', 'close']


## Step 5 — Put ownership in the search function

`closing()` guarantees that an early `return` still closes the stream.

In [20]:
def first_matching_line(
    path: Path,
    predicate: Callable[[str], bool],
    audit: list[str],
) -> str | None:
    with closing(nonempty_lines(path, audit)) as lines:
        for line in lines:
            if predicate(line):
                return line
    return None


with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "lines.txt"
    path.write_text("alpha\nbeta\ngamma\n", encoding="utf-8")

    search_audit: list[str] = []
    found = first_matching_line(path, lambda line: line.startswith("b"), search_audit)

    assert found == "beta"
    assert search_audit == ["open", "close"]

print(found, search_audit)

beta ['open', 'close']


## Step 6 — Protect cleanup from consumer exceptions

In [21]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "lines.txt"
    path.write_text("alpha\nbeta\ngamma\n", encoding="utf-8")

    consumer_error_audit: list[str] = []

    try:
        with closing(nonempty_lines(path, consumer_error_audit)) as lines:
            for line in lines:
                if line == "beta":
                    raise RuntimeError("consumer failed")
    except RuntimeError as exc:
        assert str(exc) == "consumer failed"

    assert consumer_error_audit == ["open", "close"]

print(consumer_error_audit)

['open', 'close']


## Problem 4 takeaway

Cleanup in the generator protects the resource when the frame unwinds.

The consumer that stops early is responsible for making that unwind happen immediately.

# Problem 5 — Trace close propagation through `yield from`

## Situation

A parent generator delegates to a child generator. Both have cleanup responsibilities.

We need to know whether closing the parent closes the child and in what order.

## Step 1 — Define the child

In [22]:
def child_stream(audit: list[str]) -> Iterator[int]:
    audit.append("child-open")
    try:
        yield 1
        yield 2
        yield 3
    finally:
        audit.append("child-close")

## Step 2 — Define the parent

In [23]:
def parent_stream(audit: list[str]) -> Iterator[int]:
    audit.append("parent-open")
    try:
        yield from child_stream(audit)
    finally:
        audit.append("parent-close")

## Step 3 — Close while delegation is active

The active child unwinds first, followed by the parent.

In [24]:
delegation_audit: list[str] = []
delegated = parent_stream(delegation_audit)

assert next(delegated) == 1
delegated.close()

assert delegation_audit == [
    "parent-open",
    "child-open",
    "child-close",
    "parent-close",
]

print(delegation_audit)

['parent-open', 'child-open', 'child-close', 'parent-close']


## Step 4 — Compare with normal exhaustion

In [25]:
delegation_full_audit: list[str] = []
delegated_full = parent_stream(delegation_full_audit)

assert list(delegated_full) == [1, 2, 3]
assert delegation_full_audit == [
    "parent-open",
    "child-open",
    "child-close",
    "parent-close",
]

print(delegation_full_audit)

['parent-open', 'child-open', 'child-close', 'parent-close']


## Problem 5 takeaway

`yield from` participates in close delegation.

Cleanup follows stack order: the innermost active generator closes before its parent.

# Problem 6 — Build a close-propagating transformation pipeline

## Situation

A source generator owns a resource. Mapping and filtering wrappers sit above it. The final consumer takes only a few values.

Closing the final stage should eventually close the source.

## Step 1 — Create an auditable source

In [26]:
def number_source(audit: list[str]) -> Iterator[int]:
    audit.append("source-open")
    try:
        yield from range(20)
    finally:
        audit.append("source-close")

## Step 2 — Implement a mapping stage that closes upstream

In [27]:
def map_closing(
    function: Callable[[T], U],
    source: Iterable[T],
) -> Iterator[U]:
    iterator = iter(source)
    close_upstream = getattr(iterator, "close", None)

    try:
        for item in iterator:
            yield function(item)
    finally:
        if callable(close_upstream):
            close_upstream()

## Step 3 — Implement a filtering stage that closes upstream

In [28]:
def filter_closing(
    predicate: Callable[[T], bool],
    source: Iterable[T],
) -> Iterator[T]:
    iterator = iter(source)
    close_upstream = getattr(iterator, "close", None)

    try:
        for item in iterator:
            if predicate(item):
                yield item
    finally:
        if callable(close_upstream):
            close_upstream()

## Step 4 — Close only the final pipeline stage

In [29]:
pipeline_audit: list[str] = []

source = number_source(pipeline_audit)
squares = map_closing(lambda value: value * value, source)
even_squares = filter_closing(lambda value: value % 2 == 0, squares)

observed = [next(even_squares), next(even_squares)]

assert observed == [0, 4]
assert pipeline_audit == ["source-open"]

even_squares.close()

assert pipeline_audit == ["source-open", "source-close"]

print(observed, pipeline_audit)

[0, 4] ['source-open', 'source-close']


## Step 5 — Explain the `islice` ownership trap

`islice` limits requests for values. It is not a universal resource-ownership abstraction.

Keep a reference to the source and close it explicitly.

In [30]:
islice_audit: list[str] = []
islice_source = number_source(islice_audit)

limited = list(itertools.islice(islice_source, 3))

assert limited == [0, 1, 2]
assert islice_audit == ["source-open"]

islice_source.close()

assert islice_audit == ["source-open", "source-close"]

print(limited, islice_audit)

[0, 1, 2] ['source-open', 'source-close']


## Step 6 — Package bounded consumption safely

In [31]:
def take_and_close(source: Iterable[T], count: int) -> list[T]:
    if count < 0:
        raise ValueError("count must be non-negative")

    iterator = iter(source)
    close_method = getattr(iterator, "close", None)

    try:
        return list(itertools.islice(iterator, count))
    finally:
        if callable(close_method):
            close_method()


take_audit: list[str] = []
taken = take_and_close(number_source(take_audit), 5)

assert taken == [0, 1, 2, 3, 4]
assert take_audit == ["source-open", "source-close"]

print(taken, take_audit)

[0, 1, 2, 3, 4] ['source-open', 'source-close']


## Problem 6 takeaway

A wrapper that owns its upstream iterator should forward cleanup.

This makes partial-consumption pipelines deterministic.

# Problem 7 — Build a context-managed iterator wrapper

## Situation

You want one abstraction that:

- wraps any iterable,
- forwards close if supported,
- becomes exhausted after close,
- closes on natural exhaustion,
- supports `with`,
- treats repeated close calls as harmless.

## Step 1 — Implement the complete wrapper

In [32]:
class ManagedIterator(Iterator[T]):
    def __init__(self, source: Iterable[T]) -> None:
        self._iterator = iter(source)
        self._closed = False

    @property
    def closed(self) -> bool:
        return self._closed

    def __iter__(self) -> "ManagedIterator[T]":
        return self

    def __next__(self) -> T:
        if self._closed:
            raise StopIteration

        try:
            return next(self._iterator)
        except StopIteration:
            self.close()
            raise

    def close(self) -> None:
        if self._closed:
            return

        self._closed = True
        close_method = getattr(self._iterator, "close", None)

        if callable(close_method):
            close_method()

    def __enter__(self) -> "ManagedIterator[T]":
        return self

    def __exit__(self, exc_type, exc, traceback) -> bool:
        self.close()
        return False

## Step 2 — Test an early exit from the context

In [33]:
managed_audit: list[str] = []

with ManagedIterator(number_source(managed_audit)) as managed:
    assert next(managed) == 0
    assert next(managed) == 1
    assert managed.closed is False

assert managed.closed is True
assert managed_audit == ["source-open", "source-close"]

print(managed.closed, managed_audit)

True ['source-open', 'source-close']


## Step 3 — Test repeated close and post-close iteration

In [34]:
managed.close()
managed.close()

try:
    next(managed)
except StopIteration:
    pass
else:
    raise AssertionError("Closed iterator should be exhausted")

assert managed_audit == ["source-open", "source-close"]

print("Idempotent close verified.")

Idempotent close verified.


## Step 4 — Test an iterator with no close method

In [35]:
ordinary = ManagedIterator([10, 20, 30])

assert list(ordinary) == [10, 20, 30]
assert ordinary.closed is True

print("Ordinary iterator supported.")

Ordinary iterator supported.


## Problem 7 takeaway

This wrapper adapts a basic iterator protocol to an optional cleanup protocol.

It also makes ownership visible in the public API.

# Problem 8 — Design a safe transaction coroutine

## Situation

A coroutine stages database-like changes.

A dangerous design would commit whenever `.close()` is called. Early termination could then commit incomplete work.

The protocol should instead be:

- explicit `"commit"` applies changes,
- explicit `"rollback"` discards changes,
- `close()` rolls back,
- invalid commands roll back and propagate an exception.

## Step 1 — Define the storage object and command type

In [36]:
@dataclass
class MemoryDatabase:
    values: dict[str, Any] = field(default_factory=dict)


Command = tuple[Any, ...]

## Step 2 — Implement the coroutine

In [37]:
def transaction_coroutine(
    database: MemoryDatabase,
    audit: list[str],
) -> Generator[None, Command, None]:
    staged = dict(database.values)
    committed = False
    audit.append("begin")

    try:
        while True:
            command = yield

            if not command:
                raise ValueError("command cannot be empty")

            operation = command[0]

            if operation == "set" and len(command) == 3:
                _, key, value = command
                staged[str(key)] = value
                audit.append(f"stage:set:{key}")

            elif operation == "delete" and len(command) == 2:
                _, key = command
                staged.pop(str(key), None)
                audit.append(f"stage:delete:{key}")

            elif operation == "commit" and len(command) == 1:
                database.values = staged
                committed = True
                audit.append("commit")
                return

            elif operation == "rollback" and len(command) == 1:
                audit.append("rollback:explicit")
                return

            else:
                raise ValueError(f"invalid command: {command!r}")

    except GeneratorExit:
        audit.append("rollback:close")
        raise

    except Exception:
        audit.append("rollback:error")
        raise

    finally:
        audit.append("release")
        if not committed:
            staged.clear()

## Step 3 — Prime and commit successfully

In [38]:
commit_database = MemoryDatabase({"stable": 1})
commit_audit: list[str] = []

transaction = transaction_coroutine(commit_database, commit_audit)
next(transaction)

transaction.send(("set", "new", 2))
transaction.send(("delete", "stable"))

try:
    transaction.send(("commit",))
except StopIteration:
    pass

assert commit_database.values == {"new": 2}
assert commit_audit == [
    "begin",
    "stage:set:new",
    "stage:delete:stable",
    "commit",
    "release",
]

print(commit_database.values)
print(commit_audit)

{'new': 2}
['begin', 'stage:set:new', 'stage:delete:stable', 'commit', 'release']


## Step 4 — Close before commit

The staged write must disappear.

In [39]:
close_database = MemoryDatabase({"stable": 1})
transaction_close_audit: list[str] = []

transaction_close = transaction_coroutine(
    close_database,
    transaction_close_audit,
)
next(transaction_close)
transaction_close.send(("set", "temporary", 999))
transaction_close.close()

assert close_database.values == {"stable": 1}
assert transaction_close_audit == [
    "begin",
    "stage:set:temporary",
    "rollback:close",
    "release",
]

print(close_database.values)
print(transaction_close_audit)

{'stable': 1}
['begin', 'stage:set:temporary', 'rollback:close', 'release']


## Step 5 — Trigger invalid input

In [40]:
error_database = MemoryDatabase({"stable": 1})
transaction_error_audit: list[str] = []

transaction_error = transaction_coroutine(
    error_database,
    transaction_error_audit,
)
next(transaction_error)

try:
    transaction_error.send(("unsupported",))
except ValueError as exc:
    assert "invalid command" in str(exc)

assert error_database.values == {"stable": 1}
assert transaction_error_audit == [
    "begin",
    "rollback:error",
    "release",
]

print(transaction_error_audit)

['begin', 'rollback:error', 'release']


## Problem 8 takeaway

`GeneratorExit` indicates shutdown, not successful business completion.

Commit must be an explicit domain event.

# Problem 9 — Compare a streaming generator with a generator-based context manager

## Key distinction

A streaming generator may yield many items.

A function decorated with `@contextmanager` must yield exactly once. That one suspension separates setup from teardown.

## Step 1 — Build a CSV reader context manager

In [41]:
@contextmanager
def open_csv_reader(path: Path, audit: list[str]):
    audit.append("open")
    file = path.open("r", encoding="utf-8", newline="")

    try:
        yield csv.DictReader(file)
    finally:
        file.close()
        audit.append("close")

## Step 2 — Stop reading early

Leaving the `with` block closes the file even when the reader is not exhausted.

In [42]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "items.csv"
    path.write_text(
        "name,quantity\n"
        "apple,3\n"
        "banana,5\n"
        "cherry,7\n",
        encoding="utf-8",
    )

    reader_audit: list[str] = []

    with open_csv_reader(path, reader_audit) as reader:
        first_row = next(reader)

    assert first_row == {"name": "apple", "quantity": "3"}
    assert reader_audit == ["open", "close"]

print(first_row, reader_audit)

{'name': 'apple', 'quantity': '3'} ['open', 'close']


## Step 3 — Demonstrate why multiple yields are invalid

In [43]:
@contextmanager
def invalid_context_manager():
    yield "first"
    yield "second"


try:
    with invalid_context_manager() as value:
        assert value == "first"
except RuntimeError as exc:
    invalid_context_message = str(exc)
else:
    raise AssertionError("RuntimeError was expected")

assert invalid_context_message == "generator didn't stop"

print(invalid_context_message)

generator didn't stop


## Problem 9 takeaway

Use a context manager when the API represents one ownership interval.

Use a streaming generator when the API represents a lazy sequence.

# Problem 10 — Close an asynchronous generator

## Situation

An async generator owns a resource and produces values over time.

The synchronous concepts have async counterparts:

- `close()` becomes `aclose()`,
- `closing()` becomes `aclosing()`,
- `for` becomes `async for`,
- `with` becomes `async with`.

## Step 1 — Define an async resource stream

In [44]:
async def async_number_source(audit: list[str]) -> AsyncIterator[int]:
    audit.append("async-open")
    try:
        value = 0
        while True:
            await asyncio.sleep(0)
            yield value
            value += 1
    finally:
        audit.append("async-close")

## Step 2 — Close directly with `aclose()`

In Jupyter, native top-level `await` is used instead of `asyncio.run()`.

In [45]:
async def direct_async_demo() -> tuple[list[int], list[str]]:
    audit: list[str] = []
    source = async_number_source(audit)

    values = [
        await anext(source),
        await anext(source),
    ]

    await source.aclose()
    return values, audit


direct_async_values, direct_async_audit = await direct_async_demo()

assert direct_async_values == [0, 1]
assert direct_async_audit == ["async-open", "async-close"]

print(direct_async_values, direct_async_audit)

[0, 1] ['async-open', 'async-close']


## Step 3 — Use `aclosing()` around an early break

In [46]:
async def managed_async_demo() -> tuple[list[int], list[str]]:
    audit: list[str] = []
    values: list[int] = []

    async with aclosing(async_number_source(audit)) as source:
        async for value in source:
            values.append(value)
            if value == 3:
                break

    return values, audit


managed_async_values, managed_async_audit = await managed_async_demo()

assert managed_async_values == [0, 1, 2, 3]
assert managed_async_audit == ["async-open", "async-close"]

print(managed_async_values, managed_async_audit)

[0, 1, 2, 3] ['async-open', 'async-close']


## Problem 10 takeaway

Async cleanup may itself require awaiting.

Use `aclose()` or `aclosing()` whenever an async generator is not exhausted.

# Problem 11 — Capstone: validated multi-file CSV event streaming

## Situation

Several CSV files contain events. The system should:

- open one file at a time,
- validate headers and rows,
- produce immutable event objects,
- support strict and non-strict parsing,
- stop early when a business threshold is reached,
- close the active file deterministically.

## Step 1 — Define the domain model

In [47]:
@dataclass(frozen=True)
class Event:
    event_id: str
    user_id: str
    amount: Decimal
    active: bool
    source_file: str
    line_number: int


class EventValidationError(ValueError):
    pass


EVENT_FIELDS = {"event_id", "user_id", "amount", "active"}

## Step 2 — Parse booleans safely

In [48]:
def parse_event_bool(raw: str) -> bool:
    normalized = raw.strip().lower()

    if normalized in {"true", "1"}:
        return True
    if normalized in {"false", "0"}:
        return False

    raise EventValidationError(f"invalid boolean {raw!r}")

## Step 3 — Convert a CSV row into an event

Validation errors include the source file and line number.

In [49]:
def parse_event_row(
    row: dict[str, str | None],
    *,
    source_file: str,
    line_number: int,
) -> Event:
    event_id = (row.get("event_id") or "").strip()
    user_id = (row.get("user_id") or "").strip()
    amount_text = (row.get("amount") or "").strip()
    active_text = row.get("active") or ""

    location = f"{source_file}:{line_number}"

    if not event_id:
        raise EventValidationError(f"{location}: empty event_id")

    if not user_id:
        raise EventValidationError(f"{location}: empty user_id")

    try:
        amount = Decimal(amount_text)
    except InvalidOperation as exc:
        raise EventValidationError(
            f"{location}: invalid amount {amount_text!r}"
        ) from exc

    try:
        active = parse_event_bool(active_text)
    except EventValidationError as exc:
        raise EventValidationError(f"{location}: {exc}") from exc

    return Event(
        event_id=event_id,
        user_id=user_id,
        amount=amount,
        active=active,
        source_file=source_file,
        line_number=line_number,
    )

## Step 4 — Build the single-file generator

The file is opened lazily when iteration starts and closed in `finally`.

In [50]:
def events_from_file(
    path: Path,
    audit: list[str],
    *,
    strict: bool = True,
) -> Iterator[Event]:
    audit.append(f"open:{path.name}")
    file = path.open("r", encoding="utf-8", newline="")

    try:
        reader = csv.DictReader(file)

        if reader.fieldnames is None:
            raise EventValidationError(f"{path.name}: missing header")

        actual_fields = {name.strip() for name in reader.fieldnames}

        if actual_fields != EVENT_FIELDS:
            raise EventValidationError(
                f"{path.name}: unexpected headers "
                f"{sorted(actual_fields)!r}"
            )

        for line_number, row in enumerate(reader, start=2):
            try:
                yield parse_event_row(
                    row,
                    source_file=path.name,
                    line_number=line_number,
                )
            except EventValidationError as exc:
                if strict:
                    raise
                audit.append(f"skip:{exc}")

    finally:
        file.close()
        audit.append(f"close:{path.name}")

## Step 5 — Validate one file before composing files

In [51]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "events_a.csv"
    path.write_text(
        "event_id,user_id,amount,active\n"
        "e1,u1,2.50,true\n"
        "e2,u2,3.75,0\n",
        encoding="utf-8",
    )

    one_file_audit: list[str] = []
    one_file_events = list(events_from_file(path, one_file_audit))

    assert [event.event_id for event in one_file_events] == ["e1", "e2"]
    assert one_file_events[0].amount == Decimal("2.50")
    assert one_file_events[1].active is False
    assert one_file_audit == [
        "open:events_a.csv",
        "close:events_a.csv",
    ]

print(one_file_events)
print(one_file_audit)

[Event(event_id='e1', user_id='u1', amount=Decimal('2.50'), active=True, source_file='events_a.csv', line_number=2), Event(event_id='e2', user_id='u2', amount=Decimal('3.75'), active=False, source_file='events_a.csv', line_number=3)]
['open:events_a.csv', 'close:events_a.csv']


## Step 6 — Delegate across files with `yield from`

In [52]:
def events_from_files(
    paths: Iterable[Path],
    audit: list[str],
    *,
    strict: bool = True,
) -> Iterator[Event]:
    for path in paths:
        yield from events_from_file(path, audit, strict=strict)

## Step 7 — Test full multi-file consumption

In [53]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    path_a = base / "a.csv"
    path_b = base / "b.csv"

    path_a.write_text(
        "event_id,user_id,amount,active\n"
        "a1,u1,2.00,true\n"
        "a2,u2,3.00,false\n",
        encoding="utf-8",
    )

    path_b.write_text(
        "event_id,user_id,amount,active\n"
        "b1,u3,4.00,1\n"
        "b2,u4,5.00,0\n",
        encoding="utf-8",
    )

    multi_audit: list[str] = []
    multi_events = list(events_from_files([path_a, path_b], multi_audit))

    assert [event.event_id for event in multi_events] == [
        "a1",
        "a2",
        "b1",
        "b2",
    ]

    assert multi_audit == [
        "open:a.csv",
        "close:a.csv",
        "open:b.csv",
        "close:b.csv",
    ]

print([event.event_id for event in multi_events])
print(multi_audit)

['a1', 'a2', 'b1', 'b2']
['open:a.csv', 'close:a.csv', 'open:b.csv', 'close:b.csv']


## Step 8 — Add a business threshold

The consumer owns the outer generator through `closing()` because it may stop early.

In [54]:
def collect_until_amount(
    paths: Iterable[Path],
    limit: Decimal,
    audit: list[str],
) -> tuple[Decimal, list[Event]]:
    total = Decimal("0")
    accepted: list[Event] = []

    with closing(events_from_files(paths, audit)) as events:
        for event in events:
            candidate = total + event.amount

            if candidate > limit:
                break

            total = candidate
            accepted.append(event)

    return total, accepted

## Step 9 — Stop in the middle of the second file

Closing the outer stream propagates through `yield from` to the active file generator.

In [55]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    path_a = base / "a.csv"
    path_b = base / "b.csv"

    path_a.write_text(
        "event_id,user_id,amount,active\n"
        "a1,u1,2.00,true\n"
        "a2,u2,3.00,true\n",
        encoding="utf-8",
    )

    path_b.write_text(
        "event_id,user_id,amount,active\n"
        "b1,u3,4.00,true\n"
        "b2,u4,100.00,true\n"
        "b3,u5,1.00,true\n",
        encoding="utf-8",
    )

    bounded_audit: list[str] = []
    total, accepted = collect_until_amount(
        [path_a, path_b],
        Decimal("10.00"),
        bounded_audit,
    )

    assert total == Decimal("9.00")
    assert [event.event_id for event in accepted] == ["a1", "a2", "b1"]
    assert bounded_audit == [
        "open:a.csv",
        "close:a.csv",
        "open:b.csv",
        "close:b.csv",
    ]

print(total)
print([event.event_id for event in accepted])
print(bounded_audit)

9.00
['a1', 'a2', 'b1']
['open:a.csv', 'close:a.csv', 'open:b.csv', 'close:b.csv']


## Step 10 — Test strict validation

In [56]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "bad.csv"
    path.write_text(
        "event_id,user_id,amount,active\n"
        "e1,u1,not-a-number,true\n",
        encoding="utf-8",
    )

    strict_audit: list[str] = []

    try:
        list(events_from_file(path, strict_audit, strict=True))
    except EventValidationError as exc:
        strict_message = str(exc)
    else:
        raise AssertionError("EventValidationError was expected")

    assert "bad.csv:2" in strict_message
    assert "invalid amount" in strict_message
    assert strict_audit == ["open:bad.csv", "close:bad.csv"]

print(strict_message)
print(strict_audit)

bad.csv:2: invalid amount 'not-a-number'
['open:bad.csv', 'close:bad.csv']


## Step 11 — Test non-strict validation

In [57]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "mixed.csv"
    path.write_text(
        "event_id,user_id,amount,active\n"
        "e1,u1,2.00,true\n"
        "e2,u2,invalid,false\n"
        "e3,u3,4.00,maybe\n"
        "e4,u4,5.00,1\n",
        encoding="utf-8",
    )

    relaxed_audit: list[str] = []
    relaxed_events = list(
        events_from_file(path, relaxed_audit, strict=False)
    )

    assert [event.event_id for event in relaxed_events] == ["e1", "e4"]
    assert relaxed_audit[0] == "open:mixed.csv"
    assert relaxed_audit[-1] == "close:mixed.csv"
    assert sum(item.startswith("skip:") for item in relaxed_audit) == 2

print([event.event_id for event in relaxed_events])
print(relaxed_audit)

['e1', 'e4']
['open:mixed.csv', "skip:mixed.csv:3: invalid amount 'invalid'", "skip:mixed.csv:4: invalid boolean 'maybe'", 'close:mixed.csv']


## Capstone takeaway

The capstone combines:

- lazy acquisition,
- `finally` cleanup,
- close delegation,
- explicit consumer ownership,
- safe parsing,
- strict and recoverable error policies,
- early termination without leaked files.

# Problem 12 — Build a cleanup test matrix

## Situation

Resource code should not be tested only through happy-path output.

A meaningful matrix drives the generator through every important termination mechanism.

## Step 1 — Create an instrumented generator

In [58]:
class InternalStreamError(RuntimeError):
    pass


def instrumented_stream(
    audit: list[str],
    *,
    fail_internally: bool = False,
) -> Iterator[int]:
    audit.append("entered")
    reason = "normal"

    try:
        yield 1

        if fail_internally:
            reason = "internal-error"
            raise InternalStreamError("internal failure")

        yield 2

    except GeneratorExit:
        reason = "close"
        raise

    except Exception as exc:
        if reason == "normal":
            reason = f"injected:{type(exc).__name__}"
        raise

    finally:
        audit.append(f"cleanup:{reason}")

## Step 2 — Run all lifecycle scenarios

In [59]:
matrix: dict[str, list[str]] = {}

# Close before start
audit = []
stream = instrumented_stream(audit)
stream.close()
assert audit == []
matrix["close-before-start"] = audit

# Close after start
audit = []
stream = instrumented_stream(audit)
assert next(stream) == 1
stream.close()
assert audit == ["entered", "cleanup:close"]
matrix["close-after-start"] = audit

# Normal exhaustion
audit = []
stream = instrumented_stream(audit)
assert list(stream) == [1, 2]
assert audit == ["entered", "cleanup:normal"]
matrix["normal-exhaustion"] = audit

# Internal error
audit = []
stream = instrumented_stream(audit, fail_internally=True)
assert next(stream) == 1
try:
    next(stream)
except InternalStreamError:
    pass
assert audit == ["entered", "cleanup:internal-error"]
matrix["internal-error"] = audit

# Injected error
audit = []
stream = instrumented_stream(audit)
assert next(stream) == 1
try:
    stream.throw(KeyError("caller-injected"))
except KeyError:
    pass
assert audit == ["entered", "cleanup:injected:KeyError"]
matrix["injected-error"] = audit

# Repeated close
audit = []
stream = instrumented_stream(audit)
assert next(stream) == 1
stream.close()
stream.close()
assert audit == ["entered", "cleanup:close"]
matrix["repeated-close"] = audit

for scenario, events in matrix.items():
    print(f"{scenario:>20}: {events}")

  close-before-start: []
   close-after-start: ['entered', 'cleanup:close']
   normal-exhaustion: ['entered', 'cleanup:normal']
      internal-error: ['entered', 'cleanup:internal-error']
      injected-error: ['entered', 'cleanup:injected:KeyError']
      repeated-close: ['entered', 'cleanup:close']


## Problem 12 takeaway

Treat cleanup behavior as a state machine.

Test:

- never started,
- suspended then closed,
- exhausted,
- internally failed,
- externally injected failure,
- repeated close.

# Anti-pattern clinic

## Anti-pattern 1 — Returning from `finally`

A `return` inside `finally` can suppress an active exception.

In [60]:
def suppressing_finally() -> Iterator[int]:
    try:
        yield 1
        raise ValueError("important failure")
    finally:
        return


suppression = suppressing_finally()
assert next(suppression) == 1

try:
    next(suppression)
except StopIteration:
    was_suppressed = True
else:
    was_suppressed = False

assert was_suppressed is True
print("Exception suppressed:", was_suppressed)

Exception suppressed: True


Cleanup code should normally release resources and allow the active exception or shutdown signal to continue.

## Anti-pattern 2 — Catching `BaseException` without re-raising

`BaseException` includes `GeneratorExit`, `KeyboardInterrupt`, and `SystemExit`.

If instrumentation catches it, the handler should normally re-raise immediately.

In [61]:
def cooperative_base_handler(audit: list[str]) -> Iterator[int]:
    try:
        yield 1
    except BaseException as exc:
        audit.append(type(exc).__name__)
        raise
    finally:
        audit.append("cleanup")


base_audit: list[str] = []
base_stream = cooperative_base_handler(base_audit)

assert next(base_stream) == 1
base_stream.close()

assert base_audit == ["GeneratorExit", "cleanup"]
print(base_audit)

['GeneratorExit', 'cleanup']


## Anti-pattern 3 — Assuming `break` universally closes a generator

`break` stops a loop. It does not define resource ownership for arbitrary iterators.

In [62]:
break_audit: list[str] = []
break_stream = number_source(break_audit)

for value in break_stream:
    assert value == 0
    break

assert break_audit == ["source-open"]

break_stream.close()

assert break_audit == ["source-open", "source-close"]
print(break_audit)

['source-open', 'source-close']


# Final review

## Generator shutdown checklist

Before approving a resource-owning generator, verify:

- acquisition happens inside the generator body when laziness is desired,
- release logic is in `finally`,
- an early-stopping consumer owns deterministic close,
- `GeneratorExit` is not ignored,
- the generator never yields during close,
- wrapper stages close owned upstream iterators,
- exceptions are not accidentally suppressed,
- commit is an explicit business event,
- repeated close is harmless,
- async streams use `aclose()` or `aclosing()`,
- tests cover all major termination paths,
- garbage-collection timing is not part of correctness.

## Further advanced challenges

1. Implement `merge_sorted_closing(*sources, key=...)`.
2. Add savepoints to the transaction coroutine.
3. Create an async transformation pipeline that propagates `aclose()`.
4. Record structured lifecycle event objects instead of strings.
5. Add randomized early-stop tests for every possible record position.
6. Build a retrying CSV parser with a maximum error budget.
7. Compare a custom close-aware iterator with `contextlib.ExitStack`.

# Summary

The most reliable pattern is:

1. acquire inside the generator,
2. protect the resource with `try` / `finally`,
3. cooperate with `GeneratorExit`,
4. never yield after a close request,
5. make early-stop ownership explicit,
6. propagate close through wrappers and delegation,
7. test each termination mechanism.

Used this way, generators become dependable components for streaming and resource-sensitive systems.